# EXH MMM — LOB-Level Refit (Funnel-Anchored)
**Re-fits individual LOB response curves with carryover fixed to the validated funnel learnings.**

The working grouped model validates at **Val R² 0.484** but only resolves to the *group* level.
This refit estimates **LOB-level** curves so budget can be reallocated across LOBs — the decision
stakeholders actually make — while protecting validation by spending degrees of freedom only where
the data supports them.

### Design (locked from the launch/spend diagnostic)
* **Carryover α — FIXED per funnel** (Awareness 0.751, Consideration 0.466, Conversion 0.249),
  taken from the validated grouped model. Not estimated. α drives *timing*, not marginal return, so
  fixing it costs little for reallocation and removes ~24 weakly-identified parameters.
* **All funnel cells retained** — no spend discarded. EXH-Brand, Cineclub, VIP keep every funnel
  cell; each LOB's contribution is the sum of its cells.
* **Effect γ — estimated per LOB.** Free for the 7 curve-LOBs; own-but-present for the 3
  borrow-shape LOBs; frozen tight for F&B.
* **Saturation λ (logistic) — free for the 7 curve-LOBs; inherited for the 3 borrow-shape LOBs;
  frozen for F&B.**

### Treatment table
| Treatment | LOBs | λ source |
| --- | --- | --- |
| Own curve (reallocatable) | EXH-Brand, Performance, IMAX, G&E, Cineclub, VIP, Prepaid-B2B | own |
| Borrow shape (directional) | Film Marketing, AVX, Prepaid-B2C | group/partner |
| Frozen (report only) | F&B | fixed |

Free media params: ~10 γ + 7 λ (vs the failed run's 24 α + 24 γ + 24 λ).

### Validation discipline
This refit must **re-earn** its validation. The notebook reports its Val R² head-to-head against
the grouped model's 0.484 **and** a controls-only baseline on the same split, so it's immediately
clear whether the LOB resolution came for free or at a cost.

In [0]:
%pip install pymc arviz numpyro jax jaxlib pytensor scikit-learn openpyxl netcdf4 h5netcdf --quiet

In [0]:
import os, pickle
import numpy as np
import pandas as pd
from collections import defaultdict

GROUPED_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes/"
REFIT_PATH   = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_LOB_Refit/"
OUT_PATH     = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_LOB_Decomp/"
os.makedirs(OUT_PATH, exist_ok=True)

# Grouped (validated) artifacts
with open(f"{GROUPED_PATH}response_params.pkl",       "rb") as f: response_params       = pickle.load(f)
with open(f"{GROUPED_PATH}spend_train_max_iter2.pkl", "rb") as f: spend_train_max_iter2 = pickle.load(f)
with open(f"{GROUPED_PATH}model_meta.pkl",            "rb") as f: model_meta            = pickle.load(f)
with open(f"{GROUPED_PATH}model_spec.pkl",            "rb") as f: model_spec            = pickle.load(f)
with open(f"{GROUPED_PATH}baseline_2025.pkl",         "rb") as f: baseline_2025         = pickle.load(f)
df_full = pd.read_parquet(f"{GROUPED_PATH}df_full.parquet")

ACTIVE_COMBOS = model_meta["ACTIVE_COMBOS"]
FUNNEL_COMBOS = model_meta["FUNNEL_COMBOS"]
LOB_GROUP_MAP = model_meta["LOB_GROUP_MAP"]
target_col    = model_meta["target_col"]
scale_factor  = model_spec["scale_factor"]
CONFIDENCE_TIERS = model_meta.get("CONFIDENCE_TIERS", {})

# Refit confidence tiers (curve / borrow / frozen) — the only thing we take from the refit
try:
    with open(f"{REFIT_PATH}model_meta_refit.pkl", "rb") as f: refit_meta = pickle.load(f)
    LOB_TREAT = refit_meta["LOB_TREAT"]
    print(f"Loaded refit confidence tiers for {len(LOB_TREAT)} LOBs")
except FileNotFoundError:
    LOB_TREAT = {}
    print("⚠ refit meta not found — all LOBs default to pure spend share (no tilt)")

print(f"Grouped model: {len(ACTIVE_COMBOS)} active combos | scale_factor={scale_factor:.1f}")

## 1. Configuration

In [0]:
active_contributions = {}
for combo in ACTIVE_COMBOS:
    p   = response_params[combo]
    mx  = spend_train_max_iter2[combo]
    wk  = baseline_2025[combo]["weekly_avg"]
    s   = wk / mx
    ad  = s / (1 - p["alpha"])                       # steady-state adstock
    if p["saturation"] == "hill":
        sat = (ad ** p["alpha_s"]) / (ad ** p["alpha_s"] + p["k"] ** p["alpha_s"])
    else:
        sat = 1 - np.exp(-p["lam"] * ad)
    active_contributions[combo] = p["gamma"] * sat * scale_factor * 52

total_validated = sum(active_contributions.values())
print(f"Validated total media contribution (2025): {total_validated:,.0f}")
for combo in ACTIVE_COMBOS:
    print(f"  {combo:<40} {active_contributions[combo]:>12,.0f}")

In [0]:
# Reconstruct control-only prediction from the grouped model's saved fields if present,
# else fall back to df_full's stored controls prediction.
if "controls_pred" in df_full.columns:
    control_pred = df_full["controls_pred"].values
    print("Using stored controls_pred from grouped model.")
elif "pred" in df_full.columns and "total_media" in df_full.columns:
    control_pred = (df_full["pred"] - df_full["total_media"]).values
    print("Derived controls_pred = pred − total_media.")
else:
    raise KeyError("Cannot find control prediction. Need 'controls_pred' or "
                   "('pred' and 'total_media') in df_full.")

residual_att = df_full[target_col].values - control_pred
print(f"Residual attendance: mean {residual_att.mean():,.0f} | std {residual_att.std():,.0f}")

In [0]:
media_cols = [c for c in df_full.columns if any(
    c.startswith(f"{f}_") for f in ["Awareness","Consideration","Conversion"]
) and "__" not in c and "media_" not in c]

funnel_lob_cols = defaultdict(list)
for col in media_cols:
    parts = col.split("_", 1); funnel = parts[0]; rest = parts[1]
    funnel_lob_cols[(funnel, rest[:rest.rfind("_")])].append(col)

# Weekly spend per (funnel, lob); annual 2025 spend; residual correlation
df_2025 = df_full[df_full["year"] == 2025]
lob_spend_2025, lob_tilt_corr = {}, {}
for (funnel, lob), cols in funnel_lob_cols.items():
    weekly = df_full[cols].sum(axis=1).values.astype(float)
    lob_spend_2025[(funnel, lob)] = float(df_2025[cols].sum().sum())
    # correlation only over weeks the LOB was active (else zeros dominate and bias toward 0)
    active = weekly > 0
    if active.sum() >= 5 and np.std(weekly[active]) > 0 and np.std(residual_att[active]) > 0:
        lob_tilt_corr[(funnel, lob)] = float(np.corrcoef(weekly[active], residual_att[active])[0, 1])
    else:
        lob_tilt_corr[(funnel, lob)] = 0.0   # insufficient signal → no tilt

print(f"Parsed {len(funnel_lob_cols)} (funnel, LOB) cells")
print("\nResidual-attendance correlation by cell (tilt signal):")
for k in sorted(lob_tilt_corr, key=lambda x: -abs(lob_tilt_corr[x]))[:15]:
    print(f"  {k[0]:<14} {k[1]:<34} r={lob_tilt_corr[k]:+.2f}  ${lob_spend_2025[k]:>10,.0f}")

In [0]:
TILT_CAP = 0.35   # max fractional move from spend share

def lob_display_name(raw):
    return {"EXH__Brand":"EXH - Brand","FandB":"F&B","GandE":"G&E",
            "Film_Marketing__Inform":"Film Marketing - Inform",
            "Performance__Convert_and_Upgrade":"Performance - Convert & Upgrade",
            "Premium_Experiences":"Premium Experiences - AVX",
            "Premium_Experiences__AVX":"Premium Experiences - AVX",
            "Premium_Experiences__IMAX":"Premium Experiences - IMAX",
            "Premium_Experiences__VIP":"Premium Experiences - VIP",
            "Prepaid__B2B":"Prepaid - B2B","Prepaid__B2C":"Prepaid - B2C",
            "Cineclub":"Cineclub"}.get(raw, raw)

def get_parent_combo(funnel, lob_raw):
    for group_name, lob_list in LOB_GROUP_MAP.items():
        if lob_raw in lob_list:
            combo = f"{funnel}__{group_name}"
            if combo in ACTIVE_COMBOS:
                return combo, group_name
    return None, None

# Group cells by parent combo
combo_members = defaultdict(list)
for (funnel, lob_raw), spend in lob_spend_2025.items():
    if spend <= 0:
        continue
    combo, group = get_parent_combo(funnel, lob_raw)
    if combo is None:
        combo, group = f"{funnel}__Other", "Other"
    combo_members[combo].append((funnel, lob_raw, spend))

rows = []
for combo, members in combo_members.items():
    group_total_spend   = sum(s for _, _, s in members)
    group_contribution  = active_contributions.get(combo, None)

    # Base spend shares
    base = {(f, l): (s / group_total_spend if group_total_spend > 0 else 1/len(members))
            for f, l, s in members}

    # Tilt factor per LOB (curve LOBs only; others factor = 1)
    tilt_factor = {}
    for f, l, s in members:
        disp = lob_display_name(l)
        treat = LOB_TREAT.get(disp, "curve")     # default curve if unknown
        if treat == "curve":
            r = lob_tilt_corr.get((f, l), 0.0)
            # map correlation [-1,1] to a multiplicative factor within ±TILT_CAP
            tilt_factor[(f, l)] = 1.0 + TILT_CAP * max(-1.0, min(1.0, r))
        else:
            tilt_factor[(f, l)] = 1.0           # borrow/frozen → pure spend share

    # Apply tilt, renormalise so shares sum to 1 (→ contributions sum to group total)
    raw_w = {k: base[k] * tilt_factor[k] for k in base}
    wsum  = sum(raw_w.values())
    shares = {k: raw_w[k] / wsum for k in raw_w}

    for f, l, s in members:
        disp = lob_display_name(l)
        treat = LOB_TREAT.get(disp, "curve")
        share = shares[(f, l)]
        if group_contribution is not None:
            contrib = group_contribution * share
            status  = "Decomposed"
        else:
            # combo not in validated set (e.g. '__Other') → cannot decompose a validated total
            contrib = np.nan
            status  = "No validated parent ⚠"
        rows.append({
            "Funnel": f, "LOB": disp, "Parent Combo": combo,
            "Treatment": treat,
            "Reallocatable": "YES" if treat == "curve" else
                             ("directional" if treat == "borrow" else "frozen"),
            "Annual Spend": round(s, 0),
            "Spend Share (grp)": round(base[(f, l)] * 100, 1),
            "Tilt r": round(lob_tilt_corr.get((f, l), 0.0), 2) if treat == "curve" else None,
            "Final Share (grp)": round(share * 100, 1),
            "Annual Contrib": round(contrib, 0) if not np.isnan(contrib) else None,
            "Att per $1": round(contrib / s, 2) if (not np.isnan(contrib) and s > 0) else None,
        })

decomp = pd.DataFrame(rows)
# sanity: decomposed contributions sum back to the validated total
decomp_sum = decomp["Annual Contrib"].dropna().sum()
print(f"Sum of decomposed LOB contributions : {decomp_sum:,.0f}")
print(f"Validated group total (active)      : {total_validated:,.0f}")
print(f"Difference                          : {decomp_sum - total_validated:,.0f}  "
      f"(should be ~0 if every active combo had members)")

In [0]:
show = decomp.copy()
total_spend = show["Annual Spend"].sum()
total_contrib = show["Annual Contrib"].dropna().sum()
show["Spend %"]   = (show["Annual Spend"] / total_spend * 100).round(1)
show["Contrib %"] = (show["Annual Contrib"] / total_contrib * 100).round(1)

disp = show.sort_values("Annual Contrib", ascending=False, na_position="last").copy()
disp["Annual Spend"]   = disp["Annual Spend"].apply(lambda x: f"${x:,.0f}")
disp["Annual Contrib"] = disp["Annual Contrib"].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "—")
disp["Att per $1"]     = disp["Att per $1"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
display(disp[["Funnel","LOB","Parent Combo","Reallocatable","Annual Spend","Spend %",
              "Spend Share (grp)","Tilt r","Final Share (grp)","Annual Contrib","Contrib %","Att per $1"]])

print(f"\nReallocatable (curve) LOBs carry residual-correlation tilts (capped ±{int(TILT_CAP*100)}%).")
print("Directional (borrow) and frozen LOBs use pure spend share — no tilt.")

In [0]:
moves = []
for combo, members in combo_members.items():
    if combo not in active_contributions:
        continue
    gt   = active_contributions[combo]
    gts  = sum(s for _, _, s in members)
    for f, l, s in members:
        disp_l = lob_display_name(l)
        pure   = gt * (s / gts if gts > 0 else 1/len(members))
        row    = decomp[(decomp["Funnel"] == f) & (decomp["LOB"] == disp_l) &
                        (decomp["Parent Combo"] == combo)]
        if not len(row): continue
        tilted = row["Annual Contrib"].iloc[0]
        if pd.isna(tilted): continue
        moves.append({"Funnel": f, "LOB": disp_l, "Parent": combo,
                      "Pure-share Contrib": round(pure, 0),
                      "Tilted Contrib": round(tilted, 0),
                      "Δ": round(tilted - pure, 0),
                      "Δ %": round((tilted - pure) / pure * 100, 1) if pure > 0 else 0})
moves_df = pd.DataFrame(moves).sort_values("Δ %", key=lambda s: s.abs(), ascending=False)
display(moves_df)
print(f"\nMax absolute move: {moves_df['Δ %'].abs().max():.1f}% "
      f"(capped at ±{int(TILT_CAP*100)}% before renormalisation; renorm can push slightly beyond).")

## 2. Data Loading & Feature Engineering
Identical to the grouped/training scripts so the baseline is unchanged.

In [0]:
df = pd.read_excel(FILE_PATH, sheet_name=0)
df = df.sort_values("week_date").reset_index(drop=True)

df["stat_holiday_in_week"] = (df["stat_holiday_in_week"] == "Y").astype(int)
df["is_long_weekend"]      = (df["is_long_weekend"]      == "Y").astype(int)

season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=False).astype(int)
season_dummies = season_dummies.drop(columns=["season_winter"])
df = pd.concat([df.drop(columns=["season"]), season_dummies], axis=1)

df["is_2024"]    = (df["week_date"].dt.year == 2024).astype(int)
df["is_2025"]    = (df["week_date"].dt.year == 2025).astype(int)
df["week_index"] = (df["week_date"] - df["week_date"].min()).dt.days / 7

print(f"Data {df.shape} | {df['week_date'].min().date()} → {df['week_date'].max().date()}")

In [0]:
target_col = "total_weekly_attendance"

DROP_FEATURES = [
    "national_max_temp", "national_min_temp", "national_temp_anomaly",
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity",
]
control_cols      = df.columns[3:44].tolist()
control_cols      = [c for c in control_cols if c != "season"] + season_dummies.columns.tolist()
baseline_features = [f for f in control_cols if f not in DROP_FEATURES]
baseline_features += ["is_2024", "is_2025", "week_index"]
assert len(baseline_features) == 22, f"got {len(baseline_features)}"
print(f"Baseline features: {len(baseline_features)}")

## 3. Build Funnel×LOB Cells with Treatment Assignment
Parses raw media columns, applies the v2 merges (AVX+IMAX kept *separate* here since the diagnostic
treats them distinctly; B2B/B2C kept separate as Prepaid-B2B / Prepaid-B2C), and tags each LOB with
its treatment and λ source.

In [0]:
all_cols    = df.columns.tolist()
target_idx  = all_cols.index(target_col)
media_start = all_cols.index("Awareness_Cineclub_Digital")
media_cols  = all_cols[media_start:target_idx]

def parse_media_col(col):
    parts   = col.split("_")
    funnel  = parts[0].capitalize()
    channel = parts[-1]
    lob     = "_".join(parts[1:-1])
    return funnel, lob, channel

media_meta = pd.DataFrame([parse_media_col(c)+(c,) for c in media_cols],
                          columns=["funnel","lob","channel","col_name"])

# LOB label normalisation to match the diagnostic table names
LOB_RENAME = {
    "EXH__Brand":"EXH - Brand", "FandB":"F&B", "GandE":"G&E",
    "Film_Marketing__Inform":"Film Marketing - Inform",
    "Performance__Convert_and_Upgrade":"Performance - Convert & Upgrade",
    "Premium_Experiences":"Premium Experiences - AVX",   # base Premium Experiences = AVX
    "Premium_Experiences__AVX":"Premium Experiences - AVX",
    "Premium_Experiences__IMAX":"Premium Experiences - IMAX",
    "Premium_Experiences__VIP":"Premium Experiences - VIP",
    "Prepaid__B2B":"Prepaid - B2B", "Prepaid__B2C":"Prepaid - B2C",
    "Cineclub":"Cineclub",
}
media_meta["lob_name"] = media_meta["lob"].map(lambda x: LOB_RENAME.get(x, x))

FUNNELS = ["Awareness","Consideration","Conversion"]

# ── Treatment + group assignment (from launch diagnostic) ─────────────────────
CURVE_LOBS  = ["EXH - Brand","Performance - Convert & Upgrade","Premium Experiences - IMAX",
               "G&E","Cineclub","Premium Experiences - VIP","Prepaid - B2B"]
BORROW_LOBS = {  # LOB -> group whose λ it inherits
    "Film Marketing - Inform":"Film_Marketing",
    "Premium Experiences - AVX":"Premium_Brand",
    "Prepaid - B2C":"Prepaid",
}
FROZEN_LOBS = ["F&B"]

# λ-group anchors: which group's λ each borrow-LOB inherits, and the donor LOB(s)
LAMBDA_GROUP_DONORS = {
    "Film_Marketing":["Performance - Convert & Upgrade","Premium Experiences - IMAX"],
    "Premium_Brand" :["EXH - Brand","Premium Experiences - VIP"],
    "Prepaid"       :["Prepaid - B2B"],
}

def treatment(lob):
    if lob in CURVE_LOBS:  return "curve"
    if lob in BORROW_LOBS: return "borrow"
    if lob in FROZEN_LOBS: return "frozen"
    return "curve"  # default safety

def safe(s):
    return s.replace(" ","_").replace("-","").replace("&","and").replace("/","_")

# Build cells: (funnel, lob) summed across channels
CELLS, CELL_COL, CELL_FUNNEL, CELL_LOB = [], {}, {}, {}
for funnel in FUNNELS:
    for lob in sorted(media_meta["lob_name"].unique()):
        raw = media_meta.loc[(media_meta["funnel"]==funnel)&(media_meta["lob_name"]==lob),"col_name"].tolist()
        if not raw: continue
        col = f"CELL__{funnel}__{safe(lob)}"
        df[col] = df[raw].sum(axis=1)
        if df[col].sum() <= 0:   # skip empty cells
            df.drop(columns=col, inplace=True); continue
        key = (funnel, lob)
        CELLS.append(key); CELL_COL[key]=col; CELL_FUNNEL[key]=funnel; CELL_LOB[key]=lob

LOBS = sorted({lob for (_,lob) in CELLS})
print(f"Active cells: {len(CELLS)} | LOBs: {len(LOBS)}")
for lob in LOBS:
    cells = [f for (f,l) in CELLS if l==lob]
    print(f"  {lob:<34} {treatment(lob):<7} cells: {', '.join(cells)}")

## 4. Train / Validation Split + Scaling
Same chronological split and scalers as the working model.

In [0]:
n = len(df)
n_test  = 15
n_val   = 26
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy().reset_index(drop=True)
val_df   = df.iloc[n_train : n_train + n_val].copy().reset_index(drop=True)
test_df  = df.iloc[n_train + n_val :].copy().reset_index(drop=True)

scaler_X = StandardScaler(); scaler_y = StandardScaler()
X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
X_test  = scaler_X.transform(test_df[baseline_features])
y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()
y_test  = scaler_y.transform(test_df[[target_col]]).flatten()
scale_factor = float(scaler_y.scale_[0])

# Per-cell spend max-normalised on TRAIN only; full series kept for prediction
spend_scaled, spend_train_max = {}, {}
for key in CELLS:
    col = CELL_COL[key]
    m = float(train_df[col].max()); m = m if m>0 else 1.0
    spend_train_max[key] = m
    full = df[col].values.astype(np.float64)
    spend_scaled[key] = {"train": full[:n_train]/m, "full": full/m}

print(f"Train {n_train}  ({train_df['week_date'].min().date()} → {train_df['week_date'].max().date()})")
print(f"Val   {n_val}   ({val_df['week_date'].min().date()} → {val_df['week_date'].max().date()})")
print(f"Test  {n_test}  ({test_df['week_date'].min().date()} → {test_df['week_date'].max().date()})  [held out, not used for selection]")

## 5. Response Functions

In [0]:
def geometric_adstock_fixed(x, alpha):
    '''Adstock with a FIXED (constant) alpha — computed in numpy, no sampling.'''
    out = np.zeros(len(x)); out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t-1]
    return out

def logistic_saturation(x, lam):
    return 1 - pt.exp(-lam * x)

# Pre-compute adstocked spend per cell (α is fixed, so this is deterministic)
adstocked_train = {}
for key in CELLS:
    a = ALPHA_FIXED[CELL_FUNNEL[key]]
    adstocked_train[key] = geometric_adstock_fixed(spend_scaled[key]["train"], a)
print("Adstock pre-computed with fixed α (no sampling cost).")

## 6. Model — Funnel-Anchored LOB Refit

With α fixed, only γ and λ are sampled. Structure:
* **γ per LOB** (not per cell): one effect size per LOB, shared across its funnel cells, so the LOB
  is the estimated unit. Curve & borrow LOBs get a HalfNormal γ; F&B is frozen at a tight prior.
* **λ:** curve-LOBs get own `lam_lob`; borrow-LOBs inherit their donor group's `lam_group`
  (the group λ is itself estimated from the donor LOBs' cells); F&B frozen.
* Soft media-share `Potential` retained.

In [0]:
LOB_TREAT = {lob: treatment(lob) for lob in LOBS}
LOB_CELLS = {lob: [k for k in CELLS if CELL_LOB[k]==lob] for lob in LOBS}

# Which groups need an estimated λ (donors), and their donor cells
GROUP_LAMBDAS = {g: [k for d in donors for k in LOB_CELLS.get(d,[])]
                 for g,donors in LAMBDA_GROUP_DONORS.items()}

mean_att_raw = float(train_df[target_col].mean())
MEDIA_SHARE_CAP = 0.60
share_ceiling = MEDIA_SHARE_CAP * mean_att_raw / scale_factor
FROZEN_GAMMA  = 0.05   # tight frozen effect for F&B
FROZEN_LAM    = 2.5

with pm.Model() as refit:
    intercept = pm.Normal("intercept", 0, 1)
    beta      = pm.Normal("beta", 0, 1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", 0.5)
    mu = intercept + pm.math.dot(X_train, beta)

    # Group λ for borrow donors (estimated from donor cells)
    lam_group = {g: pm.Gamma(f"lam_group_{g}", alpha=3, beta=1) for g in GROUP_LAMBDAS}

    # Per-LOB λ
    lam_lob = {}
    for lob in LOBS:
        t = LOB_TREAT[lob]
        if t == "curve":
            lam_lob[lob] = pm.Gamma(f"lam_{safe(lob)}", alpha=3, beta=1)
        elif t == "borrow":
            lam_lob[lob] = lam_group[BORROW_LOBS[lob]]   # inherit donor-group λ
        else:  # frozen
            lam_lob[lob] = FROZEN_LAM

    # Per-LOB γ
    gamma_lob = {}
    for lob in LOBS:
        t = LOB_TREAT[lob]
        if t == "frozen":
            gamma_lob[lob] = FROZEN_GAMMA
        else:
            gamma_lob[lob] = pm.HalfNormal(f"gamma_{safe(lob)}", sigma=0.3)

    # Assemble media contribution: sum over cells, γ shared within LOB
    media_terms = []
    for lob in LOBS:
        g = gamma_lob[lob]; lam = lam_lob[lob]
        for key in LOB_CELLS[lob]:
            sat  = logistic_saturation(adstocked_train[key], lam)
            term = g * sat
            media_terms.append(term)
            mu = mu + term

    total_media = pt.sum(pt.stack(media_terms, axis=0), axis=0)
    overage     = pt.maximum(total_media - share_ceiling, 0.0)
    pm.Potential("media_share_guard", -50.0 * pt.sum(overage**2))

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    n_gamma = sum(1 for l in LOBS if LOB_TREAT[l]!="frozen")
    n_lam   = sum(1 for l in LOBS if LOB_TREAT[l]=="curve") + len(GROUP_LAMBDAS)
    print(f"Free media params: {n_gamma} γ + {n_lam} λ (α fixed)")
    print("Sampling refit...")
    trace = pmjax.sample_numpyro_nuts(draws=2000, tune=3000,
                                      target_accept=0.99, chains=4, random_seed=42)

## 7. Convergence + Validation (head-to-head)
Reports R-hat/ESS/divergences, then the decisive numbers: refit Val R² vs the grouped model's
**0.484** and vs a **controls-only** baseline on the same split. If the refit can't beat
controls-only by a clear margin, the LOB media layer isn't earning its place.

In [0]:
var_names = ["intercept","beta","sigma"]
var_names += [f"gamma_{safe(l)}" for l in LOBS if LOB_TREAT[l]!="frozen"]
var_names += [f"lam_{safe(l)}" for l in LOBS if LOB_TREAT[l]=="curve"]
var_names += [f"lam_group_{g}" for g in GROUP_LAMBDAS]

summ = az.summary(trace, var_names=var_names)
summ["r_hat"] = pd.to_numeric(summ["r_hat"], errors="coerce")
print("="*55); print("CONVERGENCE"); print("="*55)
print(f"Max R-hat   : {summ['r_hat'].max():.4f}")
print(f"Min ESS     : {int(summ['ess_bulk'].min())}")
print(f"Divergences : {int(trace.sample_stats.diverging.sum().item())}")

post = trace.posterior
intercept_m = float(post["intercept"].values.mean())
beta_m      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

def lob_gamma(lob):
    if LOB_TREAT[lob]=="frozen": return FROZEN_GAMMA
    return float(post[f"gamma_{safe(lob)}"].values.mean())
def lob_lam(lob):
    t=LOB_TREAT[lob]
    if t=="curve":  return float(post[f"lam_{safe(lob)}"].values.mean())
    if t=="borrow": return float(post[f"lam_group_{BORROW_LOBS[lob]}"].values.mean())
    return FROZEN_LAM

# Full-period prediction (adstock over the FULL series, then slice)
X_all = np.vstack([X_train, X_val, X_test])
mu_ctrl = intercept_m + X_all @ beta_m
mu_media = np.zeros(n)
for lob in LOBS:
    g, lam = lob_gamma(lob), lob_lam(lob)
    for key in LOB_CELLS[lob]:
        a = ALPHA_FIXED[CELL_FUNNEL[key]]
        ad = geometric_adstock_fixed(spend_scaled[key]["full"], a)
        mu_media += g * (1 - np.exp(-lam*ad))

pred = scaler_y.inverse_transform((mu_ctrl+mu_media).reshape(-1,1)).ravel()
sl_tr = slice(0, n_train)
sl_v  = slice(n_train, n_train+n_val)
sl_te = slice(n_train+n_val, n)
act = df[target_col].values

r2_tr = r2_score(act[sl_tr], pred[sl_tr])
r2_v  = r2_score(act[sl_v],  pred[sl_v])
r2_te = r2_score(act[sl_te], pred[sl_te])
mape_tr = mean_absolute_percentage_error(act[sl_tr], pred[sl_tr])*100
mape_v  = mean_absolute_percentage_error(act[sl_v],  pred[sl_v])*100
mape_te = mean_absolute_percentage_error(act[sl_te], pred[sl_te])*100

# Controls-only baseline on the SAME val window (fit on train, score val)
from numpy.linalg import lstsq
Xtr_i = np.column_stack([np.ones(n_train), X_train])
coef,_,_,_ = lstsq(Xtr_i, y_train, rcond=None)
Xv_i = np.column_stack([np.ones(n_val), X_val])
pred_v_ctrl = scaler_y.inverse_transform((Xv_i@coef).reshape(-1,1)).ravel()
r2_v_ctrl = r2_score(act[sl_v], pred_v_ctrl)

print("\n"+"="*64)
print("VALIDATION — HEAD TO HEAD  (val = middle 26 wks; test = final 15, held out)")
print("="*64)
print(f"{'Model':<34}{'Train R²':>10}{'Val R²':>10}")
print("-"*54)
print(f"{'LOB Refit (this model)':<34}{r2_tr:>10.3f}{r2_v:>10.3f}")
print(f"{'Grouped model (reference)':<34}{'0.632':>10}{'0.484':>10}")
print(f"{'Controls-only baseline':<34}{'—':>10}{r2_v_ctrl:>10.3f}")
print(f"\nRefit MAPE: train {mape_tr:.1f}% | val {mape_v:.1f}%")
print(f"Held-out TEST (final 15 wks): R² {r2_te:.3f} | MAPE {mape_te:.1f}%")
print(f"Media contribution: {(mu_media*scale_factor).sum()/act.sum()*100:.1f}% of attendance")
verdict = ("✓ refit beats controls-only AND is near/above grouped — LOB layer earns its place"
           if (r2_v > r2_v_ctrl + 0.05 and r2_v > 0.40) else
           "⚠ refit does NOT clearly beat baselines — investigate before shipping")
print(f"\n{verdict}")

## 8. Per-LOB Curves + Reallocation Readiness
Each LOB's γ, λ, annual contribution, and marginal return — flagged by treatment so the optimizer
(next script) knows which LOBs are reallocatable vs directional vs frozen.

In [0]:
scale = scale_factor
rows = []
for lob in LOBS:
    g, lam = lob_gamma(lob), lob_lam(lob)
    spend = sum(float(df[CELL_COL[k]].sum()) for k in LOB_CELLS[lob])
    contrib = 0.0
    for key in LOB_CELLS[lob]:
        a = ALPHA_FIXED[CELL_FUNNEL[key]]
        ad = geometric_adstock_fixed(spend_scaled[key]["full"], a)
        contrib += (g*(1-np.exp(-lam*ad))).sum()*scale
    treat = LOB_TREAT[lob]
    reallocatable = "YES" if treat=="curve" else ("directional" if treat=="borrow" else "frozen")
    rows.append({"LOB":lob,"Treatment":treat,"Reallocatable":reallocatable,
                 "Funnels":", ".join(sorted({CELL_FUNNEL[k] for k in LOB_CELLS[lob]})),
                 "Annual Spend":round(spend,0),"Annual Contrib":round(contrib,0),
                 "Att per $1":round(contrib/spend,2) if spend>0 else 0,
                 "gamma":round(g,3),"lambda":round(lam,2)})
lob_curve_df = pd.DataFrame(rows).sort_values("Annual Contrib", ascending=False).reset_index(drop=True)
display(lob_curve_df)

## 9. Artifact Export (new folder)
Saves trace, scalers, per-cell normalisers, per-LOB params, fixed-α spec, and full predictions, so
the scenario/optimizer script can run against LOB-level curves. Trace written to /tmp first, then
copied (Volume FUSE can't do HDF5 random I/O).

In [0]:
print("Saving refit artifacts...")
_l="/tmp/trace_refit.nc"; _v=f"{ARTIFACT_PATH}trace_refit.nc"
for p in (_l,_v):
    if os.path.exists(p): os.remove(p)
trace.to_netcdf(_l); shutil.copy2(_l,_v); os.remove(_l)
print("✓ trace")

with open(f"{ARTIFACT_PATH}scaler_X.pkl","wb") as f: pickle.dump(scaler_X,f)
with open(f"{ARTIFACT_PATH}scaler_y.pkl","wb") as f: pickle.dump(scaler_y,f)
with open(f"{ARTIFACT_PATH}spend_train_max_cell.pkl","wb") as f: pickle.dump(spend_train_max,f)

# Per-LOB response params (means) for the optimizer
lob_params = {}
for lob in LOBS:
    lob_params[lob] = {
        "treatment": LOB_TREAT[lob],
        "reallocatable": LOB_TREAT[lob]=="curve",
        "gamma": lob_gamma(lob), "lam": lob_lam(lob),
        "alpha_by_funnel": {CELL_FUNNEL[k]: ALPHA_FIXED[CELL_FUNNEL[k]] for k in LOB_CELLS[lob]},
        "cells": [{"funnel":CELL_FUNNEL[k],"col":CELL_COL[k],
                   "train_max":spend_train_max[k]} for k in LOB_CELLS[lob]],
    }
with open(f"{ARTIFACT_PATH}lob_params.pkl","wb") as f: pickle.dump(lob_params,f)

meta = {"LOBS":LOBS,"LOB_TREAT":LOB_TREAT,"LOB_CELLS":{l:[CELL_COL[k] for k in LOB_CELLS[l]] for l in LOBS},
        "CELL_FUNNEL":{CELL_COL[k]:CELL_FUNNEL[k] for k in CELLS},
        "ALPHA_FIXED":ALPHA_FIXED,"baseline_features":baseline_features,
        "target_col":target_col,"n_train":n_train,"n_val":n_val,
        "scale_factor":scale_factor,"BORROW_LOBS":BORROW_LOBS,
        "CURVE_LOBS":CURVE_LOBS,"FROZEN_LOBS":FROZEN_LOBS}
with open(f"{ARTIFACT_PATH}model_meta_refit.pkl","wb") as f: pickle.dump(meta,f)

spec = {"saturation":"logistic","alpha":"fixed_per_funnel","scale_factor":scale_factor,
        "media_share_cap":MEDIA_SHARE_CAP,"frozen_gamma":FROZEN_GAMMA,"frozen_lam":FROZEN_LAM}
with open(f"{ARTIFACT_PATH}model_spec_refit.pkl","wb") as f: pickle.dump(spec,f)

# 2025 baseline per cell + full predictions
df_2025 = df[df["week_date"].dt.year==2025]
baseline_2025 = {}
for key in CELLS:
    col=CELL_COL[key]
    baseline_2025[CELL_COL[key]]={"annual_spend":float(df_2025[col].sum()),
                                  "weekly_avg":float(df_2025[col].mean())}
with open(f"{ARTIFACT_PATH}baseline_2025_cell.pkl","wb") as f: pickle.dump(baseline_2025,f)

df_out = df.copy(); df_out["pred"]=pred; df_out["segment"]=["train"]*n_train+["val"]*n_val
df_out.to_parquet(f"{ARTIFACT_PATH}df_full_refit.parquet", index=False)
lob_curve_df.to_csv(f"{ARTIFACT_PATH}lob_curves.csv", index=False)

print(f"\nArtifacts -> {ARTIFACT_PATH}")
for fn in sorted(os.listdir(ARTIFACT_PATH)):
    print(f"  {fn:<34}{os.path.getsize(ARTIFACT_PATH+fn)/1e6:.1f} MB")